In [42]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error

In [34]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from scipy.interpolate import interp1d
from scipy.signal import find_peaks # <-- Make sure this is imported

all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH', 'Modified SoC'] # <-- Add 'Modified SoC'
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points from 0.01 Hz to 10 kHz

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    
    # --- NEW: Get the SoC feature ---
    feat_SoC = row['Modified SoC'] # This is your new contextual feature
    
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)') 
        
        f_x = raw_df['Frequency(Hz)'].values
        R = raw_df['R(ohm)'].values
        X = raw_df['X(ohm)'].values
        Z_imag_neg = -X
        
        # --- A. Calculate 6 Peak-Finding Features ---
        feat_R_ohmic = R[-1] 
        feat_R_low = R[0]
        indices, _ = find_peaks(Z_imag_neg, prominence=1e-4) 
        
        if len(indices) == 2:
            feat_Peak_LF_Height = Z_imag_neg[indices[0]]
            feat_Peak_LF_Freq = f_x[indices[0]]
            feat_Peak_HF_Height = Z_imag_neg[indices[1]]
            feat_Peak_HF_Freq = f_x[indices[1]]
        elif len(indices) == 1:
            feat_Peak_LF_Height = Z_imag_neg[indices[0]]
            feat_Peak_LF_Freq = f_x[indices[0]]
            feat_Peak_HF_Height = 0.0 
            feat_Peak_HF_Freq = 0.0
        else:
            feat_Peak_LF_Height = 0.0
            feat_Peak_LF_Freq = 0.0
            feat_Peak_HF_Height = 0.0
            feat_Peak_HF_Freq = 0.0

        plot_features_array = np.array([
            feat_R_ohmic, feat_R_low, 
            feat_Peak_LF_Height, feat_Peak_LF_Freq,
            feat_Peak_HF_Height, feat_Peak_HF_Freq
        ])
        
        # --- NEW: B. Calculate 2 Phase-Based Features ---
        Z_phase_deg = np.angle(R + 1j * X, deg=True)
        feat_phase_min = np.min(Z_phase_deg)
        feat_phase_min_freq = f_x[np.argmin(Z_phase_deg)]
        
        phase_features_array = np.array([feat_phase_min, feat_phase_min_freq])

        # --- C. Calculate 100 Interpolated Features ---
        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        R_features = interp_R(fixed_freqs)
        X_features = interp_X(fixed_freqs)
        
        # --- D. Combine ALL features (100 + 6 + 2 + 1 = 109 features) ---
        features = np.concatenate([
            R_features, 
            X_features, 
            plot_features_array, 
            phase_features_array,
            np.array([feat_SoC]) # Add SoC
        ])
        eis_features_list.append(features)
        
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None)

# 4. Create the final aligned X and y DataFrames
r_cols = [f'R_{freq:.2f}Hz' for freq in fixed_freqs]
x_cols = [f'X_{freq:.2f}Hz' for freq in fixed_freqs]
plot_feature_names = [
    'R_ohmic', 'R_low', 'Peak_LF_Height', 'Peak_LF_Freq', 
    'Peak_HF_Height', 'Peak_HF_Freq'
]
phase_feature_names = ['Phase_Min', 'Phase_Min_Freq']
context_feature_names = ['SoC'] # Your new feature

# Combine all column names
all_feature_names = (
    r_cols + x_cols + 
    plot_feature_names + 
    phase_feature_names + 
    context_feature_names
)

X_features_df = pd.DataFrame(eis_features_list, columns=all_feature_names)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']] # (This isn't used by the GPR, but good to have)

# 5. Drop any rows that failed feature engineering
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

print(f"Data ready: X_features ({X_features_final.shape}), y_params ({y_params_final.shape})")

Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 1219.12it/s]

Data ready: X_features ((549, 109)), y_params ((549, 6))


In [35]:
X = X_features_final
y = y_params_final
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']

# --- 2. Split Data (using the same random_state) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Define the Hyperparameter Grid (Distribution) ---
# This is the range of values RandomizedSearchCV will sample from
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_features': [1.0, 0.5, 'sqrt'],
    'max_depth': [10, 20, 30, 40, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# --- 4. Initialize the Randomized Search ---
# Create the base model
rf = RandomForestRegressor(random_state=42)

rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,  # Try 50 random combinations (adjust as needed)
    cv=3,       # 3-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1,  # Use all cores
    scoring='neg_mean_squared_error' 
)

# --- 5. Run the Search (This will take some time) ---
print("Starting Hyperparameter Tuning...")
rf_random.fit(X_train, y_train)

# --- 6. Show Best Parameters ---
print("\n--- Tuning Complete ---")
print(f"Best parameters found: {rf_random.best_params_}")
print(f"Best CV (Negative MSE) score: {rf_random.best_score_:.4f}")

# --- 7. Evaluate the BEST Model on the Test Set ---
print("\n--- Evaluating Best Model on Test Set ---")
# Get the fully-tuned, best model
best_model = rf_random.best_estimator_

# Run prediction on the held-out test set
y_pred = best_model.predict(X_test)

# Calculate RMSE for each parameter separately
rmse_R0 = np.sqrt(mean_squared_error(y_test['R0'], y_pred[:, param_columns.index('R0')]))
rmse_R1 = np.sqrt(mean_squared_error(y_test['R1'], y_pred[:, param_columns.index('R1')]))
rmse_W1 = np.sqrt(mean_squared_error(y_test['W1'], y_pred[:, param_columns.index('W1')]))

print(f"Test RMSE for R0 (Tuned): {rmse_R0:.4f} Ohms")
print(f"Test RMSE for R1 (Tuned): {rmse_R1:.4f} Ohms")
print(f"Test RMSE for W1 (Tuned): {rmse_W1:.4f} Ohms")

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END bootstrap=False, max_depth=40, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   0.7s
[CV] END bootstrap=False, max_depth=40, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   0.8s
[CV] END bootstrap=False, max_depth=40, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   0.8s
[CV] END bootstrap=False, max_depth=50, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   2.7s
[CV] END bootstrap=False, max_depth=50, max_features=0.5, min_samples_leaf=4, min_samples_split=5, n_estimators=300; total time=   3.1s
[CV] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=200; total time=   0.3s
[CV] END bootstrap=False, max_depth=50, max_features=0.5, min_samples_leaf=4, min_sam

In [ ]:
X = X_features_final
y = y_params_final

# --- 2. Split Data (CRITICAL: Convert to NumPy arrays) ---
# This prevents the .dtype error you were seeing
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42
)

# --- 3. Create the base model and wrapper ---
xgb = XGBRegressor(random_state=42)
xgb_multi = MultiOutputRegressor(xgb)

# --- 4. Define the Hyperparameter Grid ---
# Must use 'estimator__' prefix for the wrapper
param_dist = {
    'estimator__n_estimators': [100, 300, 500, 700],
    'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'estimator__max_depth': [5, 10, 20, 30],
    'estimator__subsample': [0.7, 0.8, 0.9, 1.0],
    'estimator__colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

# --- 5. Initialize and run the search ---
xgb_random = RandomizedSearchCV(
    estimator=xgb_multi,
    param_distributions=param_dist,
    n_iter=50, # Try 50 random combinations
    cv=3,       
    verbose=51,
    random_state=42,
    n_jobs=-1,  # Use all cores
    scoring='neg_mean_squared_error'
)

print("\nStarting Hyperparameter Tuning for XGBoost...")
xgb_random.fit(X_train, y_train)

# --- 6. Evaluate the best XGBoost model ---
print("\n--- Tuning Complete ---")
print(f"Best parameters found: {xgb_random.best_params_}")
print(f"Best CV (Negative MSE) score: {xgb_random.best_score_:.4f}")

print("\n--- Evaluating Best XGBoost Model on Test Set ---")
best_model = xgb_random.best_estimator_
y_pred = best_model.predict(X_test)

# --- 7. Calculate RMSE using integer indexing ---
rmse_R0 = np.sqrt(mean_squared_error(y_test[:, param_columns.index('R0')], y_pred[:, param_columns.index('R0')]))
rmse_R1 = np.sqrt(mean_squared_error(y_test[:, param_columns.index('R1')], y_pred[:, param_columns.index('R1')]))
rmse_W1 = np.sqrt(mean_squared_error(y_test[:, param_columns.index('W1')], y_pred[:, param_columns.index('W1')]))

print(f"Test RMSE for R0 (Tuned XGB): {rmse_R0:.4f} Ohms")
print(f"Test RMSE for R1 (Tuned XGB): {rmse_R1:.4f} Ohms")
print(f"Test RMSE for W1 (Tuned XGB): {rmse_W1:.4f} Ohms")


Starting Hyperparameter Tuning for XGBoost...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END estimator__colsample_bytree=0.8, estimator__learning_rate=0.1, estimator__max_depth=30, estimator__n_estimators=700, estimator__subsample=0.7; total time=  12.1s
[CV] END estimator__colsample_bytree=0.8, estimator__learning_rate=0.1, estimator__max_depth=30, estimator__n_estimators=700, estimator__subsample=0.7; total time=  13.7s
[CV] END estimator__colsample_bytree=0.8, estimator__learning_rate=0.05, estimator__max_depth=20, estimator__n_estimators=300, estimator__subsample=0.8; total time=  15.5s
[CV] END estimator__colsample_bytree=0.8, estimator__learning_rate=0.05, estimator__max_depth=20, estimator__n_estimators=300, estimator__subsample=0.8; total time=  17.6s
[CV] END estimator__colsample_bytree=0.8, estimator__learning_rate=0.05, estimator__max_depth=20, estimator__n_estimators=300, estimator__subsample=0.8; total time=  18.0s
[CV] END estimator__colsample_byt

In [44]:
X = X_features_final
y = y_params_final
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']

# --- 2. Split Data (CRITICAL: Convert to NumPy arrays) ---
# We use .values to prevent data type errors during fitting
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42
)

# --- 3. Define the Kernels to try ---
kernel_1 = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)
kernel_2 = ConstantKernel(1.0) * RBF(length_scale=10.0) + WhiteKernel(noise_level=0.1)
kernel_3 = ConstantKernel(1.0) * RBF(length_scale=5.0) + WhiteKernel(noise_level=1.0)

# --- 4. Create the GPR Pipeline ---
# The pipeline scales the data and then fits the multi-output GPR
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', MultiOutputRegressor(
        GaussianProcessRegressor(random_state=42, normalize_y=True)
    ))
])

# --- 5. Define the Hyperparameter Grid for GPR ---
# We access the GPR's parameters inside the pipeline
param_dist = {
    'model__estimator__kernel': [kernel_1, kernel_2, kernel_3],
    'model__estimator__alpha': [1e-1, 1e-2, 1e-3, 1e-4]
}

# --- 6. Initialize the Randomized Search ---
# GPR is very slow, so we use a low n_iter
gpr_random = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,  # Keep this low
    cv=3,       
    verbose=51,
    random_state=42,
    n_jobs=-1,  # Use all cores
    scoring='neg_mean_squared_error' 
)

# --- 7. Run the Search (This will be slow) ---
print("Starting Hyperparameter Tuning for GPR...")
gpr_random.fit(X_train, y_train)

# --- 8. Show Best Parameters ---
print("\n--- Tuning Complete ---")
print(f"Best parameters found: {gpr_random.best_params_}")
print(f"Best CV (Negative MSE) score: {gpr_random.best_score_:.4f}")

# --- 9. Evaluate the BEST Model on the Test Set ---
print("\n--- Evaluating Best GPR Model on Test Set ---")
best_model = gpr_random.best_estimator_
y_pred = best_model.predict(X_test)

# --- 10. (MODIFIED) Calculate RMSE using integer indexing ---
# y_test is now a NumPy array
rmse_R0 = np.sqrt(mean_squared_error(y_test[:, param_columns.index('R0')], y_pred[:, param_columns.index('R0')]))
rmse_R1 = np.sqrt(mean_squared_error(y_test[:, param_columns.index('R1')], y_pred[:, param_columns.index('R1')]))
rmse_W1 = np.sqrt(mean_squared_error(y_test[:, param_columns.index('W1')], y_pred[:, param_columns.index('W1')]))

print(f"Test RMSE for R0 (Tuned GPR): {rmse_R0:.4f} Ohms")
print(f"Test RMSE for R1 (Tuned GPR): {rmse_R1:.4f} Ohms")
print(f"Test RMSE for W1 (Tuned GPR): {rmse_W1:.4f} Ohms")

Starting Hyperparameter Tuning for GPR...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV 1/3; 1/10] START model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1)
[CV 2/3; 1/10] START model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1)
[CV 3/3; 1/10] START model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1)
[CV 1/3; 2/10] START model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1)
[CV 2/3; 2/10] START model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1)
[CV 3/3; 2/10] START model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1)
[CV 1/3; 3/10] START model__estimator__alpha=0.1, model__estimator__kernel=1**2 * RB

/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[CV 3/3; 1/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.087 total time=   1.6s
[CV 3/3; 3/10] START model__estimator__alpha=0.1, model__estimator__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1)
[CV 1/3; 1/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.166 total time=   1.8s
[CV 1/3; 4/10] START model__estimator__alpha=0.001, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1)
[CV 2/3; 1/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.016 total time=   1.8s
[CV 2/3; 4/10] START model__estimator__alpha=0.001, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1)
[CV 1/3; 2/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_sc

/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[CV 2/3; 2/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1);, score=-0.016 total time=   2.3s
[CV 2/3; 5/10] START model__estimator__alpha=0.01, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1)
[CV 2/3; 3/10] END model__estimator__alpha=0.1, model__estimator__kernel=1**2 * RBF(length_scale=1) + WhiteKernel(noise_level=1);, score=-0.016 total time=   2.3s
[CV 3/3; 5/10] START model__estimator__alpha=0.01, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1)


...

[CV 1/3; 8/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1);, score=-0.166 total time=   1.5s
[CV 3/3; 10/10] START model__estimator__alpha=0.001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1)


/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[CV 2/3; 8/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1);, score=-0.016 total time=   1.5s
[CV 1/3; 9/10] END model__estimator__alpha=0.01, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.166 total time=   1.3s
[CV 3/3; 8/10] END model__estimator__alpha=0.0001, model__estimator__kernel=1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1);, score=-0.087 total time=   1.5s
[CV 2/3; 9/10] END model__estimator__alpha=0.01, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.016 total time=   1.5s
[CV 1/3; 10/10] END model__estimator__alpha=0.001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.166 total time=   1.1s
[CV 2/3; 10/10] END model__estimator__alpha=0.001, model__estimator__kernel=1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.1);, score=-0.016 total time=   1

/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



--- Tuning Complete ---
Best parameters found: {'model__estimator__kernel': 1**2 * RBF(length_scale=5) + WhiteKernel(noise_level=1), 'model__estimator__alpha': 0.01}
Best CV (Negative MSE) score: -0.0897

--- Evaluating Best GPR Model on Test Set ---
Test RMSE for R0 (Tuned GPR): 0.0105 Ohms
Test RMSE for R1 (Tuned GPR): 0.0956 Ohms
Test RMSE for W1 (Tuned GPR): 0.0007 Ohms


In [ ]:
# --- 1. Define your DataFrame ---
df = all_battery_data_EIS.copy()

param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']
target_column = 'SoH'

df_cleaned = df.dropna(subset=param_columns + [target_column])

X = df_cleaned[param_columns]
y = df_cleaned[target_column]

print(f"Training Model 2 (SoH Predictor) with {len(df_cleaned)} data points.")

# --- 4. Split Data into Training and Testing Sets ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 5. Initialize and Train the Model ---
# n_estimators=100 is a good default
Model_SoH = RandomForestRegressor(n_estimators=100, random_state=42)

print("Fitting Model_SoH...")
Model_SoH.fit(X_train, y_train)

# --- 6. Evaluate the Model ---
print("Evaluating Model_SoH...")
y_pred = Model_SoH.predict(X_test)

# Calculate RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Convert RMSE to a percentage SoH error
print(f"Test RMSE: {rmse*100:.2f}% SoH")

# --- 7. (Optional) Show Feature Importance ---
print("\n--- Feature Importance (which params matter most) ---")
importance = pd.Series(Model_SoH.feature_importances_, index=param_columns)
print(importance.sort_values(ascending=False))

Training Model 2 (SoH Predictor) with 549 data points.
Fitting Model_SoH...
Evaluating Model_SoH...
Test RMSE: 2.59% SoH

--- Feature Importance (which params matter most) ---
R0        0.928789
L1        0.030171
CPE1_1    0.011040
R1        0.010499
CPE1_0    0.010283
W1        0.009218
dtype: float64


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# --- (Assuming 'all_battery_data_EIS' and 'col_names' exist) ---
# Make sure your 'circuit_model' (e.g., L1-R0-p(R1,CPE1)-W1)
# parameters are correctly listed here.
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']
# ===================================================================
# PHASE 0: DATA & FEATURE ENGINEERING (Prerequisite)
# ===================================================================

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH']
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points from 0.01 Hz to 10 kHz

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)')
        
        f_x = raw_df['Frequency(Hz)'].values
        R = raw_df['R(ohm)'].values
        X = raw_df['X(ohm)'].values
        
        # Create interpolation functions
        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        
        # Get features at our fixed frequencies
        R_features = interp_R(fixed_freqs)
        X_features = interp_X(fixed_freqs)
        
        # Combine R and X to make one long feature vector
        features = np.concatenate([R_features, X_features])
        eis_features_list.append(features)
        
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None) # Add a placeholder

# 4. Create the final aligned X and y DataFrames
r_cols = [f'R_{freq:.2f}Hz' for freq in fixed_freqs]
x_cols = [f'X_{freq:.2f}Hz' for freq in fixed_freqs]

X_features_df = pd.DataFrame(eis_features_list, columns=r_cols + x_cols)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']] # Use [['SoH']] to keep it as a DataFrame

# 5. Drop any rows that failed feature engineering
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

print(f"Data ready: X_features ({X_features_final.shape}), y_params ({y_params_final.shape}), y_soh ({y_soh_final.shape})")

# ===================================================================
# PHASE 1: PYTORCH SETUP (Dataset & Model Architecture)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components ---")

# 1. Scale features
# Scaling is CRITICAL for neural networks
scaler_X = StandardScaler()
scaler_y_params = StandardScaler()

X_scaled = scaler_X.fit_transform(X_features_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final)
y_soh_values = y_soh_final.values # No scaling needed for SoH (already 0-1)

# 2. Train/Test Split
(X_train, X_val, 
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_scaled, y_params_scaled, y_soh_values, test_size=0.2, random_state=42
)

# 3. Custom PyTorch Dataset
class EISDataset(Dataset):
    def __init__(self, features, params, soh):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y_params = torch.tensor(params, dtype=torch.float32)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_val, y_params_val, y_soh_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 4. Define the Multi-Head Model
class MultiHeadEISModel(nn.Module):
    def __init__(self, input_size, num_params):
        super(MultiHeadEISModel, self).__init__()
        
        # Shared Body
        self.shared_body = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Head 1: Parameter Prediction
        self.param_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_params) # No activation (linear output for regression)
        )
        
        # Head 2: SoH Prediction
        self.soh_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1), # No activation (linear output for regression)
            nn.Sigmoid() # Add Sigmoid to bound output between 0 and 1
        )

    def forward(self, x):
        # Pass input through the shared body
        shared_output = self.shared_body(x)
        
        # Pass shared output to each head
        params_pred = self.param_head(shared_output)
        soh_pred = self.soh_head(shared_output)
        
        return params_pred, soh_pred

# ===================================================================
# PHASE 2: MODEL TRAINING
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

# 1. Setup Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

INPUT_SIZE = X_features_final.shape[1]
NUM_PARAMS = y_params_final.shape[1]
NUM_EPOCHS = 50 # Increase this for better results
LEARNING_RATE = 0.001

model = MultiHeadEISModel(INPUT_SIZE, NUM_PARAMS).to(device)
loss_params_fn = nn.MSELoss()
loss_soh_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Define your loss weights
alpha = 0.7 # Weight for SoH loss (as per your example)
beta = 0.3  # Weight for Parameter loss

# 3. Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    
    for features, true_params, true_soh in train_loader:
        features = features.to(device)
        true_params = true_params.to(device)
        true_soh = true_soh.to(device)
        
        # --- Forward Pass ---
        pred_params, pred_soh = model(features)
        
        # --- Calculate Combined Loss ---
        loss_params = loss_params_fn(pred_params, true_params)
        loss_soh = loss_soh_fn(pred_soh, true_soh) # .view(-1, 1) not needed, shape is correct
        
        loss_total = (alpha * loss_soh) + (beta * loss_params)
        
        # --- Backward Pass ---
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        
        total_train_loss += loss_total.item()

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    total_val_soh_loss = 0
    total_val_params_loss = 0
    
    with torch.no_grad():
        for features, true_params, true_soh in val_loader:
            features = features.to(device)
            true_params = true_params.to(device)
            true_soh = true_soh.to(device)
            
            pred_params, pred_soh = model(features)
            
            loss_params = loss_params_fn(pred_params, true_params)
            loss_soh = loss_soh_fn(pred_soh, true_soh)
            loss_total = (alpha * loss_soh) + (beta * loss_params)
            
            total_val_loss += loss_total.item()
            total_val_soh_loss += loss_soh.item()
            total_val_params_loss += loss_params.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    avg_soh_loss = total_val_soh_loss / len(val_loader)
    avg_params_loss = total_val_params_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} (SoH: {avg_soh_loss:.6f}, Params: {avg_params_loss:.6f})")

print("\n--- Training Complete ---")

# --- 4. Final Evaluation on Test Set ---
model.eval()
with torch.no_grad():
    # Predict on the entire validation set
    all_features = torch.tensor(X_val, dtype=torch.float32).to(device)
    true_params = torch.tensor(y_params_val, dtype=torch.float32).to(device)
    true_soh = torch.tensor(y_soh_val, dtype=torch.float32).to(device)
    
    pred_params_scaled, pred_soh = model(all_features)
    
    # --- De-scale the predictions to be human-readable ---
    pred_params = scaler_y_params.inverse_transform(pred_params_scaled.cpu().numpy())
    
    # --- Calculate Final Errors ---
    soh_rmse = np.sqrt(mean_squared_error(true_soh.cpu().numpy(), pred_soh.cpu().numpy()))
    
    print(f"\n--- Final Model Evaluation ---")
    print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")
    
    # Calculate RMSE for each parameter
    for i, name in enumerate(param_columns):
        # Compare the i-th column of y_params_val with the i-th column of pred_params
        param_rmse = np.sqrt(mean_squared_error(y_params_val[:, i], pred_params[:, i]))
        print(f"  - {name} RMSE: {param_rmse:.4f}")

--- Phase 0: Preparing Data & Engineering Features ---


Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 2256.35it/s]


Data ready: X_features ((549, 100)), y_params ((549, 6)), y_soh ((549, 1))

--- Phase 1: Setting up PyTorch components ---

--- Phase 2: Starting Model Training ---
Using device: cpu
Epoch [01/50] - Train Loss: 0.360790 | Val Loss: 0.319636 (SoH: 0.054766, Params: 0.937665)
Epoch [02/50] - Train Loss: 0.246785 | Val Loss: 0.276951 (SoH: 0.013127, Params: 0.892541)
Epoch [03/50] - Train Loss: 0.212234 | Val Loss: 0.268031 (SoH: 0.010668, Params: 0.868544)
Epoch [04/50] - Train Loss: 0.191069 | Val Loss: 0.258103 (SoH: 0.006163, Params: 0.845962)
Epoch [05/50] - Train Loss: 0.181970 | Val Loss: 0.257562 (SoH: 0.003666, Params: 0.849986)
Epoch [06/50] - Train Loss: 0.176470 | Val Loss: 0.249688 (SoH: 0.003264, Params: 0.824677)
Epoch [07/50] - Train Loss: 0.169156 | Val Loss: 0.251190 (SoH: 0.002358, Params: 0.831799)
Epoch [08/50] - Train Loss: 0.178638 | Val Loss: 0.243534 (SoH: 0.001906, Params: 0.807333)
Epoch [09/50] - Train Loss: 0.166811 | Val Loss: 0.252567 (SoH: 0.001899, Params: